In [ ]:
# =========================================================
# Klasifikačná úloha (Task 2) – multilingválna klasifikácia článkov
# ---------------------------------------------------------
# Cieľ: natrénovať model, ktorý vie z nadpisu + perexu automaticky určiť kategóriu článku.
# V kóde riešim aj praktické veci ako reproducibilita, nevyvážené triedy, tokenizácia a detailná analýza chýb.
# =========================================================

import re
import html
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
    EarlyStoppingCallback
)
import evaluate
import torch
import torch.nn as nn

# -----------------------------
# A) Reprodukovateľnosť
# -----------------------------
# Nastavujem seed naprieč knižnicami tak, aby boli výsledky opakovateľné.
# Je to dôležité hlavne pri porovnávaní zmien v experimentoch (aby rozdiely neboli len “náhodou”).
RANDOM_SEED = 42
set_seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Len pre pohodlnejší výpis dát v notebooku (dlhšie texty sa nestrihnú hneď v preview)
pd.set_option("display.max_colwidth", 160)

# -----------------------------
# B) Načítanie dát
# -----------------------------
# Načítavam samostatné CSV pre jednotlivé typy článkov.
# Ak súbory nemajú vyplnený label (alebo ho nemajú vôbec), doplním ho z názvu datasetu.
DATA_FILES = [
    ("C:/Users/Kristián/Downloads/reaction.csv",   "Reaction"),
    ("C:/Users/Kristián/Downloads/prematch.csv",   "Pre-Match"),
    ("C:/Users/Kristián/Downloads/interview.csv",  "Interview"),
    ("C:/Users/Kristián/Downloads/injuries.csv",   "Injuries"),
    ("C:/Users/Kristián/Downloads/transfers.csv",  "Transfers"),
    ("C:/Users/Kristián/Downloads/report.csv",     "Report"),
]

def clean_text(s: str) -> str:
    """
    Jemné čistenie textu:
    - HTML unescape (napr. &amp; -> &)
    - zjednotenie whitespace
    Nechcem robiť agresívne NLP úpravy (stemming/lemmatizácia), lebo to pri XLM-R zvyčajne nepomáha.
    """
    if s is None:
        return ""
    s = str(s)
    s = html.unescape(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

dfs = []
for path, default_label in DATA_FILES:
    df = pd.read_csv(path)

    # Ak stĺpec label chýba, doplním ho podľa súboru.
    # Ak existuje, ošetrím NaN a prázdne hodnoty.
    if "label" not in df.columns:
        df["label"] = default_label
    else:
        df["label"] = df["label"].fillna(default_label)
        df.loc[df["label"].astype(str).str.strip().eq(""), "label"] = default_label

    # Niektoré súbory nemusia mať perex – doplním prázdny, aby pipeline zostala jednotná.
    if "perex" not in df.columns:
        df["perex"] = ""

    # Používam len title, perex, label (aby som minimalizoval šum a riešil to konzistentne).
    df = df[["title", "perex", "label"]].copy()
    df["title"] = df["title"].fillna("").map(clean_text)
    df["perex"] = df["perex"].fillna("").map(clean_text)

    dfs.append(df)

# Spojím všetky triedy do jedného dataframe
data = pd.concat(dfs, ignore_index=True)

# -----------------------------
# C) Jednoducha heuristika (feature tags)
# -----------------------------
# Pri článkoch existujú signály, ktoré často pomáhajú rozlíšiť podobné triedy:
# - reporty často obsahujú minúty (17. min / 90+2’)
# - rozhovory/reakcie často obsahujú citácie (úvodzovky)
# - injuries/transfers majú typickú slovnú zásobu
# Nechcem robiť rule-based klasifikáciu, ale iba pridať modelu "hint" tokeny na začiatok textu.
MINUTE_RE = re.compile(r"\b\d{1,2}\.\s*(min|minute|minúta|minúte|min)\b", re.IGNORECASE)
APOSTROPHE_MIN_RE = re.compile(r"\b\d{1,2}\s*['’]\b")

INJURY_RE = re.compile(r"\b(zran|zranen|zranil|nedohral|mimo\s+hry|absenc|maród|marod|rekonval)\w*", re.IGNORECASE)
TRANSFER_RE = re.compile(r"\b(prestup|transfer|hosťovan|hostovan|posila|posil|podpis|zmluv|kontrakt|odchádza|prichádza)\w*", re.IGNORECASE)

def build_text_with_tags(title: str, perex: str) -> str:
    """
    Z title + perex skladám jeden textový vstup pre klasifikátor.
    Navyše pridávam:
    - štruktúrne tokeny [TITLE] a [PEREX]
    - feature tagy typu [HAS_MINUTES], [HAS_QUOTES], [HAS_INJURY], [HAS_TRANSFER]
    Tagy dávam na začiatok, aby ich model videl ešte pred truncationom a aby slúžili ako rýchla nápoveda.
    """
    t = title or ""
    p = perex or ""
    joined = f"{t} {p}".strip()

    tags = []

    # Citácie často signalizujú Interview alebo Reaction
    if ('"' in joined) or ("„" in joined) or ("“" in joined) or ("”" in joined) or ("'" in joined):
        tags.append("[HAS_QUOTES]")

    # Minúty udalostí sú typické pre match reporty
    if MINUTE_RE.search(joined) or APOSTROPHE_MIN_RE.search(joined):
        tags.append("[HAS_MINUTES]")

    # Lexika typická pre injuries / transfers
    if INJURY_RE.search(joined):
        tags.append("[HAS_INJURY]")

    if TRANSFER_RE.search(joined):
        tags.append("[HAS_TRANSFER]")

    tag_str = " ".join(tags)
    if tag_str:
        tag_str += " "

    # Vstup explicitne štruktúrujem, aby model videl, čo je titulok a čo je perex
    return (tag_str + "[TITLE] " + t + " [PEREX] " + p).strip()

# Vytvorím finálny textový stĺpec pre model
data["text"] = data.apply(lambda r: build_text_with_tags(r["title"], r["perex"]), axis=1)

# Ošetrím prípad, že by niečo ostalo prázdne
data = data[data["text"].str.len() > 0].copy()

# Kontrolný výpis: počet riadkov + rozdelenie tried
print("Rows:", len(data))
print("Label distribution:\n", data["label"].value_counts())

# -----------------------------
# D) Stratified split
# -----------------------------
# Použijem stratifikované delenie, aby si train/val/test zachovali podobné zastúpenie tried.
# Je to dôležité hlavne pre menšie triedy, kde by náhodný split mohol vyrobiť skreslené metriky.
train_val_df, test_df = train_test_split(
    data, test_size=0.15, random_state=RANDOM_SEED, stratify=data["label"]
)
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1765,  # ~0.15 z celého datasetu (lebo 0.1765 * 0.85 ≈ 0.15)
    random_state=RANDOM_SEED,
    stratify=train_val_df["label"]
)

print("\nTrain:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# -----------------------------
# E) Label mapping
# -----------------------------
# Labely mapujem na integer id (tak to očakáva HF Trainer).
labels = sorted(data["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

def df_to_hf(df: pd.DataFrame) -> Dataset:
    """
    Konverzia pandas -> HuggingFace Dataset.
    Labely mapujem na čísla podľa label2id.
    """
    tmp = df[["text", "label"]].copy()
    tmp["label"] = tmp["label"].map(label2id).astype(int)
    return Dataset.from_pandas(tmp, preserve_index=False)

hf_train = df_to_hf(train_df)
hf_val   = df_to_hf(val_df)
hf_test  = df_to_hf(test_df)

# -----------------------------
# F) Model/tokenizer
# -----------------------------
# Volím xlm-roberta-base, pretože je to robustný multilingválny transformer vhodný pre klasifikáciu textu.
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Pridám si vlastné špeciálne tokeny, ktoré používam vo vstupe (TITLE/PEREX a feature tagy).
# Následne musím zväčšiť embeddingy modelu, aby nové tokeny mali vlastné reprezentácie.
SPECIALS = ["[TITLE]", "[PEREX]", "[HAS_QUOTES]", "[HAS_MINUTES]", "[HAS_INJURY]", "[HAS_TRANSFER]"]
tokenizer.add_special_tokens({"additional_special_tokens": SPECIALS})

# MAX_LEN dávam 256 ako kompromis medzi výkonom a rýchlosťou na CPU.
MAX_LEN = 256

def tokenize(batch):
    # Tokenizácia s truncation, aby sme dodržali max_length.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

# Dôležité: odstránim "text" stĺpec po tokenizácii, aby collator nedostával stringy.
hf_train = hf_train.map(tokenize, batched=True, remove_columns=["text"])
hf_val   = hf_val.map(tokenize, batched=True, remove_columns=["text"])
hf_test  = hf_test.map(tokenize, batched=True, remove_columns=["text"])

# Nastavím výstup do torch tensorov, aby to HF Trainer vedel priamo trénovať.
hf_train.set_format(type="torch")
hf_val.set_format(type="torch")
hf_test.set_format(type="torch")

# Data collator spraví dynamické paddingovanie v rámci batchu (efektívnejšie ako fixná dĺžka).
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Inicializujem klasifikačný model pre počet tried v datasete.
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# Po pridaní special tokenov upravím veľkosť embedding vrstvy.
model.resize_token_embeddings(len(tokenizer))

# -----------------------------
# G) Metrics
# -----------------------------
# Vyhodnocujem accuracy + F1 macro + F1 weighted.
# Macro F1 používam hlavne kvôli nevyváženým triedam – nech to neoptimalizuje iba “majority class”.
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, y_true = eval_pred
    y_pred = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=y_pred, references=y_true)["accuracy"]
    f1m = f1_metric.compute(predictions=y_pred, references=y_true, average="macro")["f1"]
    f1w = f1_metric.compute(predictions=y_pred, references=y_true, average="weighted")["f1"]

    return {"accuracy": acc, "f1_macro": f1m, "f1_weighted": f1w}

# -----------------------------
# H) Class weights (TRAIN only) + custom Trainer
# -----------------------------
# Dataset je nevyvážený, preto nerobím oversampling (duplicitné príklady môžu zhoršiť generalizáciu),
# ale riešim to váhovanou loss funkciou (vyššia váha pre minoritné triedy).
train_counts = train_df["label"].value_counts()

# Počty prevediem do array podľa poradia label id.
counts_arr = np.zeros(len(labels), dtype=np.float32)
for lbl, cnt in train_counts.items():
    counts_arr[label2id[lbl]] = float(cnt)

# Klasické inverse-frequency váhy:
# weight_i = total / (num_classes * count_i)
total = counts_arr.sum()
weights = total / (len(labels) * np.maximum(counts_arr, 1.0))
weights_t = torch.tensor(weights, dtype=torch.float)

class WeightedLossTrainer(Trainer):
    """
    Vlastný Trainer len kvôli tomu, aby som vedel použiť váhovanú CrossEntropyLoss.
    Ostatné správanie nechávam rovnaké ako HF Trainer.
    """
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Labely si vytiahnem zvlášť, lebo model ich nepotrebuje ako input argument.
        labels_in = inputs.get("labels")
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits

        # Váhy presuniem na rovnaké zariadenie (CPU/GPU) ako logits.
        cw = self.class_weights.to(logits.device)

        # Váhovaná cross entropy loss
        loss_fct = nn.CrossEntropyLoss(weight=cw)
        loss = loss_fct(logits, labels_in)

        return (loss, outputs) if return_outputs else loss

# -----------------------------
# I) TrainingArguments
# -----------------------------
# Nastavenie je zvolené s ohľadom na CPU:
# - menší train batch + gradient accumulation (efektívne väčší batch bez nárokov na RAM)
# - väčší eval batch pre rýchlejšie vyhodnocovanie
training_args = TrainingArguments(
    output_dir="./xlmr_article_category_v5_fast",
    learning_rate=2e-5,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # efektívne batch size 16
    per_device_eval_batch_size=32,

    num_train_epochs=8,
    weight_decay=0.01,
    seed=RANDOM_SEED,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    # vyberiem najlepší checkpoint podľa macro F1 (kvôli menším triedam)
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    report_to="none",

    # mierny label smoothing pre stabilnejšie učenie (menej prehnane isté predikcie)
    label_smoothing_factor=0.03,

    dataloader_num_workers=0,
    dataloader_pin_memory=False,  # na CPU odstráni zbytočné varovanie/pinning

    remove_unused_columns=True
)

# Zostavím Trainer (s váhovanou loss) + early stopping, aby som sa vyhol zbytočnému preučeniu.
trainer = WeightedLossTrainer(
    class_weights=weights_t,
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# -----------------------------
# J) Train + Evaluate
# -----------------------------
# Natrénujem model a vyhodnotím ho na validačnej aj testovacej množine.
trainer.train()

print("\n--- VAL metrics ---")
print(trainer.evaluate(hf_val))

print("\n--- TEST metrics ---")
print(trainer.evaluate(hf_test))

# -----------------------------
# K) Detailed TEST report + error analysis
# -----------------------------
# Okrem agregovaných metrík si vytlačím:
# - classification report per trieda
# - confusion matrix
# - top confusions (najčastejšie zámeny)
# - indexy zle klasifikovaných vzoriek pre manuálnu kontrolu
pred_out = trainer.predict(hf_test)
test_logits = pred_out.predictions
test_preds = np.argmax(test_logits, axis=-1)

# hf_test["label"] je HF datasets Column (nie klasický list), preto ho premapujem cez list(...)
y_true = [id2label[int(i)] for i in list(hf_test["label"])]
y_pred = [id2label[int(i)] for i in test_preds.tolist()]

print("\nTEST classification report:\n", classification_report(y_true, y_pred, digits=3))
print("\nTEST confusion matrix (labels order):\n", labels)
cm = confusion_matrix(y_true, y_pred, labels=labels)
print(cm)

# Zozbieram páry (true_label, pred_label) pre zle klasifikované prípady
pairs = defaultdict(list)
for i, (t, p) in enumerate(zip(y_true, y_pred)):
    if t != p:
        pairs[(t, p)].append(i)

# Zoradím zámeny podľa počtu výskytov – aby som videl, kde model najviac zlyháva
conf_counts = sorted(((k, len(v)) for k, v in pairs.items()), key=lambda x: -x[1])

print("\nTop confusions:")
for (t, p), c in conf_counts[:10]:
    print(f"{t} -> {p}: {c}")

# Indexy prvých pár chybných predikcií (praktické pri manuálnom debugovaní konkrétnych textov)
mis_idx = [i for i, (t, p) in enumerate(zip(y_true, y_pred)) if t != p]
print("\nMisclassified sample indices (first 10):", mis_idx[:10])


Rows: 1500
Label distribution:
 label
Reaction     1000
Pre-Match     100
Interview     100
Injuries      100
Transfers     100
Report        100
Name: count, dtype: int64

Train: 1049 Val: 226 Test: 225


Map: 100%|██████████| 225/225 [00:00<00:00, 11579.08 examples/s]
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Kristián\AppData\Local\Temp\ipykernel_38156\2634605388.py:228: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,1.793000,1.738731,0.699115,0.242097,0.587585
2,1.513500,1.066734,0.707965,0.490312,0.685440
3,1.091900,0.793692,0.809735,0.730544,0.810178
4,0.651100,0.762406,0.818584,0.720441,0.813450
5,0.509900,0.656965,0.867257,0.790933,0.863819
6,0.403500,0.684190,0.867257,0.787559,0.861395
7,0.256900,0.658001,0.871681,0.809190,0.870742
8,0.221400,0.732425,0.871681,0.794433,0.867666



--- VAL metrics ---


{'eval_loss': 0.6580013632774353, 'eval_accuracy': 0.8716814159292036, 'eval_f1_macro': 0.8091904961623593, 'eval_f1_weighted': 0.8707424796120232, 'eval_runtime': 16.6968, 'eval_samples_per_second': 13.536, 'eval_steps_per_second': 0.479, 'epoch': 8.0}

--- TEST metrics ---
{'eval_loss': 0.7458519339561462, 'eval_accuracy': 0.8444444444444444, 'eval_f1_macro': 0.7619624885844125, 'eval_f1_weighted': 0.8443823108700066, 'eval_runtime': 19.8393, 'eval_samples_per_second': 11.341, 'eval_steps_per_second': 0.403, 'epoch': 8.0}

TEST classification report:
               precision    recall  f1-score   support

    Injuries      0.846     0.733     0.786        15
   Interview      0.667     0.533     0.593        15
   Pre-Match      0.692     0.600     0.643        15
    Reaction      0.905     0.893     0.899       150
      Report      0.583     0.933     0.718        15
   Transfers      0.933     0.933     0.933        15

    accuracy                          0.844       225
   mac